# V15 · First ensemble: XGBoost + LightGBM + CatBoost blend

**Held-out test RMSLE: 0.19596** · blended OOF RMSLE: 0.19343

- Three gradient-boosting models (6,000 trees, lr 0.02) trained with 5-fold CV on `log1p(TargetValue)`
- Blend weights found by grid search on out-of-fold predictions: **XGBoost 0.65 · LightGBM 0.30 · CatBoost 0.05**
- Treats both 1000 and 1001 as missing-year placeholders; median-imputes the hour meter

**Caveat:** the target encoding is fitted on the **full** training set before CV, so the OOF score (0.1934) is optimistic compared with the test score (0.1960). V16 fixes this.

> This is the original notebook as run on Kaggle. Data paths point to `/kaggle/input/...`; to run it locally, change them to your data folder. Code and outputs are unchanged.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/sample_submission.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/metadata.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv


In [2]:
trainData = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv")
testData = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv")


/tmp/ipykernel_16/681260659.py:1: DtypeWarning: Columns (38,39) have mixed types. Specify dtype option on import or set low_memory=False.
  trainData = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv")


In [3]:
trainData.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138701 entries, 0 to 138700
Data columns (total 50 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   TransactionID              138701 non-null  int64  
 1   TargetValue                138701 non-null  float64
 2   AssetID                    138701 non-null  int64  
 3   ProductConfigID            138701 non-null  int64  
 4   DataOriginCode             138701 non-null  object 
 5   VendorPartnerID            138701 non-null  object 
 6   ManufactureYear            138701 non-null  int64  
 7   OperationalHoursMeter      82058 non-null   float64
 8   UtilizationTier            50475 non-null   object 
 9   TransactionDate            138701 non-null  object 
 10  Spec_FullDescriptor        138701 non-null  object 
 11  Spec_BaseClass             138701 non-null  object 
 12  Spec_SubClass              101716 non-null  object 
 13  Spec_ReleaseSeries         32

In [4]:
trainData.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
TransactionID,138701.0,NaN,NaN,NaN,2186275.950556,1128496.619347,1139309.0,1465754.0,1751884.0,2372139.0,6333410.0
TargetValue,138701.0,NaN,NaN,NaN,41521.874047,26361.125948,7500.0,20500.0,35000.0,57000.0,142000.0
AssetID,138701.0,NaN,NaN,NaN,1204814.320423,528737.593449,8.0,1007930.0,1246956.0,1495817.0,2478476.0
ProductConfigID,138701.0,NaN,NaN,NaN,7286.421922,7542.421537,39.0,1693.0,3852.0,11873.0,37209.0
DataOriginCode,138701,6,ADI149,59541,NaN,NaN,NaN,NaN,NaN,NaN,NaN
VendorPartnerID,138701,31,UJF,67828,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ManufactureYear,138701.0,NaN,NaN,NaN,1891.640853,306.992001,1001.0,1990.0,1998.0,2004.0,2015.0
OperationalHoursMeter,82058.0,NaN,NaN,NaN,4553.718943,27922.04485,0.0,0.0,1620.0,5001.0,1729600.0
UtilizationTier,50475,3,Medium,24588,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TransactionDate,138701,3676,2010-02-16,1159,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
trainData.head()

,TransactionID,TargetValue,AssetID,ProductConfigID,DataOriginCode,VendorPartnerID,ManufactureYear,OperationalHoursMeter,UtilizationTier,TransactionDate,...,col20,col21,col22,col23,col24,col25,col27,col28,col29,col30
0,1139309,57000.0,117665,88,ACH138,GTX,1997,4640.0,Low,2005-03-26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Standard,Conventional
1,1139316,26500.0,1001282,4616,ACH138,GTX,2005,508.0,Low,2009-12-18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1139317,21000.0,772709,1948,ACH138,GTX,1994,11540.0,High,2005-08-26,...,Steel,None or Unspecified,None or Unspecified,None or Unspecified,None or Unspecified,Double,NaN,NaN,NaN,NaN
3,1139322,27000.0,902010,3550,ACH138,GTX,2002,4883.0,High,2006-11-17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1139333,21500.0,1036259,36014,ACH138,GTX,2009,302.0,Low,2010-08-27,...,Rubber,None or Unspecified,None or Unspecified,None or Unspecified,None or Unspecified,Double,NaN,NaN,NaN,NaN


In [6]:
trainData.nunique()

TransactionID                138701
TargetValue                     675
AssetID                      120111
ProductConfigID                3594
DataOriginCode                    6
VendorPartnerID                  31
ManufactureYear                  68
OperationalHoursMeter         14643
UtilizationTier                   3
TransactionDate                3676
Spec_FullDescriptor            3505
Spec_BaseClass                 1249
Spec_SubClass                   147
Spec_ReleaseSeries              113
Spec_VariantModifier            132
AssetScaleFactor                  6
FunctionalClassification         65
RegionCode                       53
InventoryGroupCategory            6
InventoryGroupDescription         6
col1                              4
CabinType                         3
Forks                             2
col3                              3
col4                              2
DrivetrainType                    8
col5                              2
col6                        

In [7]:
trainData.isna().mean()*100

TransactionID                 0.000000
TargetValue                   0.000000
AssetID                       0.000000
ProductConfigID               0.000000
DataOriginCode                0.000000
VendorPartnerID               0.000000
ManufactureYear               0.000000
OperationalHoursMeter        40.838206
UtilizationTier              63.608770
TransactionDate               0.000000
Spec_FullDescriptor           0.000000
Spec_BaseClass                0.000000
Spec_SubClass                26.665273
Spec_ReleaseSeries           76.282795
Spec_VariantModifier         62.269919
AssetScaleFactor             29.711394
FunctionalClassification      0.000000
RegionCode                    0.000000
InventoryGroupCategory        0.000000
InventoryGroupDescription     0.000000
col1                         75.641127
CabinType                     0.000000
Forks                        78.861724
col3                         78.909309
col4                         92.515555
DrivetrainType           

In [8]:
metadata = pd.DataFrame({
    'dtype': trainData.dtypes.astype(str),
    'nulls': trainData.isnull().sum(),
    'null_pct': (trainData.isnull().mean() * 100).round(2),
    'unique': trainData.nunique(),
    'sample_value': trainData.iloc[0]
})
metadata

,dtype,nulls,null_pct,unique,sample_value
TransactionID,int64,0,0.00,138701,1139309
TargetValue,float64,0,0.00,675,57000.0
AssetID,int64,0,0.00,120111,117665
ProductConfigID,int64,0,0.00,3594,88
DataOriginCode,object,0,0.00,6,ACH138
VendorPartnerID,object,0,0.00,31,GTX
ManufactureYear,int64,0,0.00,68,1997
OperationalHoursMeter,float64,56643,40.84,14643,4640.0
UtilizationTier,object,88226,63.61,3,Low
TransactionDate,object,0,0.00,3676,2005-03-26


In [9]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_log_error

from catboost import CatBoostRegressor

In [10]:
train_ids = trainData["TransactionID"].copy()
test_ids = testData["TransactionID"].copy()

In [11]:
trainData["InventoryGroupCategory"].value_counts()

InventoryGroupCategory
TEX    71018
MG     23405
WL     18872
TTT    14959
BL     10381
SSL       66
Name: count, dtype: int64

In [12]:
trainData = trainData.drop(columns=["TransactionID", "InventoryGroupCategory"], errors="ignore")

In [13]:
trainData["TransactionDate"] = pd.to_datetime(trainData["TransactionDate"])
trainData["TransactionYear"] = trainData["TransactionDate"].dt.year
trainData["TransactionMonth"] = trainData["TransactionDate"].dt.month
trainData["TransactionQuarter"] = trainData["TransactionDate"].dt.quarter
trainData["TransactionDay"] = trainData["TransactionDate"].dt.day
trainData["TransactionDayOfWeek"] = trainData["TransactionDate"].dt.dayofweek


trainData = trainData.drop(columns="TransactionDate")

In [14]:
trainData["ManufactureYearMissing"] = trainData["ManufactureYear"].isin([1000, 1001]).astype(int)
trainData["ManufactureYear"] = trainData["ManufactureYear"].replace([1000, 1001], np.nan)
trainData["AssetAge"] = trainData["TransactionYear"] - trainData["ManufactureYear"]
trainData.loc[trainData["AssetAge"] < 0, "AssetAge"] = np.nan

In [15]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
trainData["HoursMissing"] = trainData["OperationalHoursMeter"].isna().astype(int)
trainData["OperationalHoursMeter"] = imputer.fit_transform(trainData[["OperationalHoursMeter"]])

In [16]:
trainData["OperationalHoursMeter"].isna()

0         False
1         False
2         False
3         False
4         False
          ...  
138696    False
138697    False
138698    False
138699    False
138700    False
Name: OperationalHoursMeter, Length: 138701, dtype: bool

In [17]:
trainData["OperationalHoursLog"] = np.log1p(trainData["OperationalHoursMeter"])
trainData["HoursPerYear"] = trainData["OperationalHoursMeter"] / (trainData["AssetAge"] + 1)
trainData["Missing_BaseClass"] = trainData["Spec_BaseClass"].isna().astype(int)

In [18]:
unknown_cols = ["UtilizationTier", "Spec_SubClass", "Spec_ReleaseSeries", "Spec_VariantModifier", "AssetScaleFactor", "DrivetrainType", "Forks"]
for col in unknown_cols:
    if col in trainData.columns:
        trainData[col] = trainData[col].fillna("Unknown")

In [19]:
machine_cols = ["col1","col3","col4","col5","col6","col7","col8","col9","col10","col11","col12","col13",
                "col14","col15","col16","col18","col19","col20","col21","col22","col23","col24","col25",
                "col27","col28","col29","col30"]
for col in machine_cols:
    if col in trainData.columns:
        trainData[col] = trainData[col].fillna("Not Applicable")

In [20]:
testData = testData.drop(columns=["TransactionID", "InventoryGroupCategory"], errors="ignore")

testData["TransactionDate"] = pd.to_datetime(testData["TransactionDate"])
testData["TransactionYear"] = testData["TransactionDate"].dt.year
testData["TransactionMonth"] = testData["TransactionDate"].dt.month
testData["TransactionQuarter"] = testData["TransactionDate"].dt.quarter
testData["TransactionDay"] = testData["TransactionDate"].dt.day
testData["TransactionDayOfWeek"] = testData["TransactionDate"].dt.dayofweek
testData = testData.drop(columns="TransactionDate")

testData["ManufactureYearMissing"] = testData["ManufactureYear"].isin([1000, 1001]).astype(int)
testData["ManufactureYear"] = testData["ManufactureYear"].replace([1000, 1001], np.nan)
testData["AssetAge"] = testData["TransactionYear"] - testData["ManufactureYear"]
testData.loc[testData["AssetAge"] < 0, "AssetAge"] = np.nan


testData["HoursMissing"] = testData["OperationalHoursMeter"].isna().astype(int)
testData["OperationalHoursMeter"] = imputer.transform(testData[["OperationalHoursMeter"]])

testData["OperationalHoursLog"] = np.log1p(testData["OperationalHoursMeter"])
testData["HoursPerYear"] = testData["OperationalHoursMeter"] / (testData["AssetAge"] + 1)
testData["Missing_BaseClass"] = testData["Spec_BaseClass"].isna().astype(int)

for col in unknown_cols:
    if col in testData.columns:
        testData[col] = testData[col].fillna("Unknown")

for col in machine_cols:
    if col in testData.columns:
        testData[col] = testData[col].fillna("Not Applicable")

In [21]:
print(trainData.shape)
print(testData.shape)

(138701, 58)
(15000, 57)


In [22]:
metadata = pd.DataFrame({
    'dtype': trainData.dtypes.astype(str),
    'nulls': trainData.isnull().sum(),
    'null_pct': (trainData.isnull().mean() * 100).round(2),
    'unique': trainData.nunique(),
    'sample_value': trainData.iloc[0]
})
metadata

,dtype,nulls,null_pct,unique,sample_value
TargetValue,float64,0,0.00,675,57000.0
AssetID,int64,0,0.00,120111,117665
ProductConfigID,int64,0,0.00,3594,88
DataOriginCode,object,0,0.00,6,ACH138
VendorPartnerID,object,0,0.00,31,GTX
ManufactureYear,float64,14719,10.61,67,1997.0
OperationalHoursMeter,float64,0,0.00,14643,4640.0
UtilizationTier,object,0,0.00,4,Low
Spec_FullDescriptor,object,0,0.00,3505,950FII
Spec_BaseClass,object,0,0.00,1249,950


In [23]:
import numpy as np


target_cols = [
    "Spec_FullDescriptor", "Spec_BaseClass", "Spec_SubClass",
    "Spec_ReleaseSeries", "Spec_VariantModifier", 
    "FunctionalClassification", "RegionCode", "VendorPartnerID"
]


trainData["log_target"] = np.log1p(trainData["TargetValue"])
global_mean = trainData["log_target"].mean()

for col in target_cols:
    
    col_means = trainData.groupby(col)["log_target"].mean()
    
    
    trainData[col] = trainData[col].map(col_means)
    testData[col] = testData[col].map(col_means)
    
   
    trainData[col] = trainData[col].fillna(global_mean)
    testData[col] = testData[col].fillna(global_mean)


trainData = trainData.drop(columns=["log_target"])

In [24]:
from sklearn.preprocessing import OrdinalEncoder


cat_cols = trainData.select_dtypes(include=["object"]).columns.tolist()


encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

trainData[cat_cols] = encoder.fit_transform(trainData[cat_cols].astype(str))
testData[cat_cols] = encoder.transform(testData[cat_cols].astype(str))

In [25]:
trainData = trainData.drop(columns=["AssetID"], errors="ignore")
testData = testData.drop(columns=["AssetID"], errors="ignore")

print("Train shape:", trainData.shape)
print("Test shape:", testData.shape)

Train shape: (138701, 57)
Test shape: (15000, 56)


In [26]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_log_error


y = trainData["TargetValue"]

X = trainData.drop(columns=["TargetValue"])
X_test = testData.copy()


kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


In [27]:
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

In [28]:
models = {

    "xgb": xgb.XGBRegressor(
        n_estimators=6000,
        learning_rate=0.02,
        max_depth=8,
        min_child_weight=5,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=3,
        reg_alpha=0.1,
        tree_method="hist",
        early_stopping_rounds=300,
        random_state=42,
        n_jobs=-1
    ),


    "lgb": lgb.LGBMRegressor(
        n_estimators=6000,
        learning_rate=0.02,
        num_leaves=80,
        min_child_samples=30,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=3,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    ),


    "cat": CatBoostRegressor(
        iterations=6000,
        learning_rate=0.02,
        depth=8,
        l2_leaf_reg=5,
        loss_function="RMSE",
        random_seed=42,
        early_stopping_rounds=300,
        verbose=0
    )

}

In [29]:
oof = {
    name: np.zeros(len(X))
    for name in models
}

test_preds = {
    name: np.zeros(len(X_test))
    for name in models
}

In [30]:
for fold,(tr,val) in enumerate(kf.split(X)):

    print("\nFold",fold+1)
    X_train = X.iloc[tr]
    X_val = X.iloc[val]

    y_train = np.log1p(y.iloc[tr])
    y_val = np.log1p(y.iloc[val])
    
    for name,model in models.items():
        if name=="lgb":
            model.fit(
                X_train,
                y_train,
                eval_set=[(X_val,y_val)],
                callbacks=[
                    lgb.early_stopping(
                        300,
                        verbose=False
                    )
                ]          )

        elif name=="cat":
            model.fit(
                X_train,
                y_train,
                eval_set=(X_val,y_val),
                use_best_model=True
            )
        else:
            model.fit(
                X_train,
                y_train,
                eval_set=[(X_val,y_val)],
                verbose=0
            )

        pred = model.predict(X_val)

        oof[name][val] = pred

        test_preds[name] += (
            model.predict(X_test)/5
        )
        score = np.sqrt(
            mean_squared_log_error(
                y.iloc[val],
                np.expm1(pred)
            )
        )
        print(
            name,
            round(score,5)
        )


Fold 1
xgb 0.19196
lgb 0.19364
cat 0.19882

Fold 2
xgb 0.19303
lgb 0.19483
cat 0.20073

Fold 3
xgb 0.19543
lgb 0.19651
cat 0.2021

Fold 4
xgb 0.19553
lgb 0.19659
cat 0.20262

Fold 5
xgb 0.19397
lgb 0.19522
cat 0.20093


In [31]:
best_score = 999
best_weights = None

for wx in np.arange(0,1.05,0.05):
    for wl in np.arange(0,1-wx+0.05,0.05):
        wc =1-wx-wl
        blend =(
            wx*oof["xgb"]
            +
            wl*oof["lgb"]
            +
            wc*oof["cat"]
        )
        score = np.sqrt(
            mean_squared_log_error(
                y,
                np.expm1(blend)
            )
        )
        if score < best_score:

            best_score = score
            best_weights = (wx,wl,wc)

print("Best score:",best_score)
print("Weights:",best_weights)

Best score: 0.19343423271355892
Weights: (np.float64(0.65), np.float64(0.30000000000000004), np.float64(0.04999999999999993))


In [32]:
wx,wl,wc = best_weights

final = (
    wx*test_preds["xgb"]
    +
    wl*test_preds["lgb"]
    +
    wc*test_preds["cat"]
)
final = np.maximum(
    np.expm1(final),
    0
)


In [33]:
submission = pd.DataFrame({
    "TransactionID":test_ids,
    "TargetValue":final
})
submission.to_csv(
    "submission.csv",
    index=False
)
submission.head(10)

,TransactionID,TargetValue
0,1139307,69087.101381
1,1139419,81478.044440
2,1139482,30796.103654
3,1139522,18329.055428
4,1139684,12384.483123
5,1139773,28493.920280
6,1139779,19843.505395
7,1139793,43406.048826
8,1139814,70501.674288
9,1139839,14829.805365


In [34]:
"""import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_log_error


y = trainData["TargetValue"]
X = trainData.drop(columns=["TargetValue"])
X_test = testData.copy()


kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds_lgb = np.zeros(len(X))
test_preds_lgb = np.zeros(len(X_test))
cv_scores_lgb = []"""

'import lightgbm as lgb\nfrom sklearn.model_selection import KFold\nfrom sklearn.metrics import mean_squared_log_error\n\n\ny = trainData["TargetValue"]\nX = trainData.drop(columns=["TargetValue"])\nX_test = testData.copy()\n\n\nkf = KFold(n_splits=5, shuffle=True, random_state=42)\n\noof_preds_lgb = np.zeros(len(X))\ntest_preds_lgb = np.zeros(len(X_test))\ncv_scores_lgb = []'

In [35]:
"""fold_number = 1

for train_index, valid_index in kf.split(X):
    print("\nFold", fold_number)

    X_train = X.iloc[train_index]
    X_valid = X.iloc[valid_index]

    
    y_train = np.log1p(y.iloc[train_index])
    y_valid = np.log1p(y.iloc[valid_index])

    
    model = lgb.LGBMRegressor(
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=8,
        num_leaves=64,           
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    
    model.fit(
        X_train, 
        y_train,
        eval_set=[(X_valid, y_valid)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=200, verbose=False),
            lgb.log_evaluation(period=500)
        ]
    )

    
    valid_pred = model.predict(X_valid)
    valid_pred = np.expm1(valid_pred)
    valid_pred = np.maximum(valid_pred, 0)

    
    test_pred = model.predict(X_test)
    test_pred = np.expm1(test_pred)
    test_pred = np.maximum(test_pred, 0)

    
    y_true_valid = y.iloc[valid_index]
    score = np.sqrt(mean_squared_log_error(y_true_valid, valid_pred))
    cv_scores_lgb.append(score)

    
    oof_preds_lgb[valid_index] = valid_pred
    test_preds_lgb = test_preds_lgb + (test_pred / 5)

    print("Fold RMSLE :", round(score, 5))
    
    fold_number = fold_number + 1"""

'fold_number = 1\n\nfor train_index, valid_index in kf.split(X):\n    print("\nFold", fold_number)\n\n    X_train = X.iloc[train_index]\n    X_valid = X.iloc[valid_index]\n\n    \n    y_train = np.log1p(y.iloc[train_index])\n    y_valid = np.log1p(y.iloc[valid_index])\n\n    \n    model = lgb.LGBMRegressor(\n        n_estimators=3000,\n        learning_rate=0.03,\n        max_depth=8,\n        num_leaves=64,           \n        subsample=0.8,\n        colsample_bytree=0.8,\n        random_state=42,\n        n_jobs=-1\n    )\n\n    \n    model.fit(\n        X_train, \n        y_train,\n        eval_set=[(X_valid, y_valid)],\n        callbacks=[\n            lgb.early_stopping(stopping_rounds=200, verbose=False),\n            lgb.log_evaluation(period=500)\n        ]\n    )\n\n    \n    valid_pred = model.predict(X_valid)\n    valid_pred = np.expm1(valid_pred)\n    valid_pred = np.maximum(valid_pred, 0)\n\n    \n    test_pred = model.predict(X_test)\n    test_pred = np.expm1(test_pred)\n

In [36]:
"""print("\nCV Results (LightGBM)")
for i in range(len(cv_scores_lgb)):
    print("Fold " + str(i+1) + ": " + str(round(cv_scores_lgb[i], 5)))

mean_score = sum(cv_scores_lgb) / len(cv_scores_lgb)
print("\nMean RMSLE :", round(mean_score, 5))


oof_safe = np.maximum(oof_preds_lgb, 0)
final_oof_lgb = np.sqrt(mean_squared_log_error(y, oof_safe))
print("OOF RMSLE :", round(final_oof_lgb, 5))


"""

'print("\nCV Results (LightGBM)")\nfor i in range(len(cv_scores_lgb)):\n    print("Fold " + str(i+1) + ": " + str(round(cv_scores_lgb[i], 5)))\n\nmean_score = sum(cv_scores_lgb) / len(cv_scores_lgb)\nprint("\nMean RMSLE :", round(mean_score, 5))\n\n\noof_safe = np.maximum(oof_preds_lgb, 0)\nfinal_oof_lgb = np.sqrt(mean_squared_log_error(y, oof_safe))\nprint("OOF RMSLE :", round(final_oof_lgb, 5))\n\n\n'

In [37]:
"""submission_lgb = pd.DataFrame()
submission_lgb["TransactionID"] = test_ids
submission_lgb["TargetValue"] = test_preds_lgb

submission_lgb.to_csv("submission.csv", index=False)
print(submission_lgb.head())"""

'submission_lgb = pd.DataFrame()\nsubmission_lgb["TransactionID"] = test_ids\nsubmission_lgb["TargetValue"] = test_preds_lgb\n\nsubmission_lgb.to_csv("submission.csv", index=False)\nprint(submission_lgb.head())'

In [38]:
"""import xgboost as xgb



y = trainData["TargetValue"]
X = trainData.drop(columns=["TargetValue"])
X_test = testData.copy()

print("Train shape:", X.shape)
print("Test shape:", X_test.shape)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_predictions = np.zeros(len(X))
test_predictions = np.zeros(len(X_test))
cv_scores = []"""

'import xgboost as xgb\n\n\n\ny = trainData["TargetValue"]\nX = trainData.drop(columns=["TargetValue"])\nX_test = testData.copy()\n\nprint("Train shape:", X.shape)\nprint("Test shape:", X_test.shape)\n\nkf = KFold(n_splits=5, shuffle=True, random_state=42)\n\noof_predictions = np.zeros(len(X))\ntest_predictions = np.zeros(len(X_test))\ncv_scores = []'

In [39]:
"""fold_number = 1

for train_index, valid_index in kf.split(X):
    print("\nFold", fold_number)

    X_train = X.iloc[train_index]
    X_valid = X.iloc[valid_index]

    
    y_train = np.log1p(y.iloc[train_index])
    y_valid = np.log1p(y.iloc[valid_index])

  
    model = xgb.XGBRegressor(
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        early_stopping_rounds=200,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train, 
        y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=500
    )

    
    valid_pred = model.predict(X_valid)
    valid_pred = np.expm1(valid_pred)
    valid_pred = np.maximum(valid_pred, 0)

  
    test_pred = model.predict(X_test)
    test_pred = np.expm1(test_pred)
    test_pred = np.maximum(test_pred, 0)

    
    y_true_valid = y.iloc[valid_index]
    score = np.sqrt(mean_squared_log_error(y_true_valid, valid_pred))
    cv_scores.append(score)

    
    oof_predictions[valid_index] = valid_pred
    test_predictions = test_predictions + (test_pred / 5)

    print("Fold RMSLE :", round(score, 5))
    
    fold_number = fold_number + 1"""

'fold_number = 1\n\nfor train_index, valid_index in kf.split(X):\n    print("\nFold", fold_number)\n\n    X_train = X.iloc[train_index]\n    X_valid = X.iloc[valid_index]\n\n    \n    y_train = np.log1p(y.iloc[train_index])\n    y_valid = np.log1p(y.iloc[valid_index])\n\n  \n    model = xgb.XGBRegressor(\n        n_estimators=3000,\n        learning_rate=0.03,\n        max_depth=8,\n        subsample=0.8,\n        colsample_bytree=0.8,\n        tree_method="hist",\n        early_stopping_rounds=200,\n        random_state=42,\n        n_jobs=-1\n    )\n\n    model.fit(\n        X_train, \n        y_train,\n        eval_set=[(X_valid, y_valid)],\n        verbose=500\n    )\n\n    \n    valid_pred = model.predict(X_valid)\n    valid_pred = np.expm1(valid_pred)\n    valid_pred = np.maximum(valid_pred, 0)\n\n  \n    test_pred = model.predict(X_test)\n    test_pred = np.expm1(test_pred)\n    test_pred = np.maximum(test_pred, 0)\n\n    \n    y_true_valid = y.iloc[valid_index]\n    score = np.

In [40]:
"""print("CV Results")
for i in range(len(cv_scores)):
    print("Fold " + str(i+1) + ": " + str(round(cv_scores[i], 5)))

mean_score = sum(cv_scores) / len(cv_scores)
print("Mean RMSLE :", round(mean_score, 5))


oof_preds_safe = np.maximum(oof_predictions, 0)
final_oof = np.sqrt(mean_squared_log_error(y, oof_preds_safe))
print("OOF RMSLE :", round(final_oof, 5))
"""

'print("CV Results")\nfor i in range(len(cv_scores)):\n    print("Fold " + str(i+1) + ": " + str(round(cv_scores[i], 5)))\n\nmean_score = sum(cv_scores) / len(cv_scores)\nprint("Mean RMSLE :", round(mean_score, 5))\n\n\noof_preds_safe = np.maximum(oof_predictions, 0)\nfinal_oof = np.sqrt(mean_squared_log_error(y, oof_preds_safe))\nprint("OOF RMSLE :", round(final_oof, 5))\n'

In [41]:
"""submission = pd.DataFrame()
submission["TransactionID"] = test_ids
submission["TargetValue"] = test_predictions

submission.to_csv("submission.csv", index=False)
print(submission.head())"""

'submission = pd.DataFrame()\nsubmission["TransactionID"] = test_ids\nsubmission["TargetValue"] = test_predictions\n\nsubmission.to_csv("submission.csv", index=False)\nprint(submission.head())'

In [42]:
"""kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_predictions = np.zeros(len(X))
test_predictions = np.zeros(len(X_test))
cv_scores = []"""

'kf = KFold(n_splits=5, shuffle=True, random_state=42)\n\noof_predictions = np.zeros(len(X))\ntest_predictions = np.zeros(len(X_test))\ncv_scores = []'

In [43]:
"""fold_number = 1

for train_index, valid_index in kf.split(X):
    print("\nFold", fold_number)

    X_train = X.iloc[train_index]
    X_valid = X.iloc[valid_index]

    y_train = np.log1p(y.iloc[train_index])
    y_valid = np.log1p(y.iloc[valid_index])

    model = CatBoostRegressor(
        iterations=5000,
        learning_rate=0.03,
        depth=8,
        loss_function="RMSE",
        random_seed=42,
        early_stopping_rounds=200,
        verbose=500
    )

    model.fit(
        X_train,
        y_train,
        eval_set=(X_valid, y_valid),
        use_best_model=True
    )

   
    valid_pred = model.predict(X_valid)
    valid_pred = np.expm1(valid_pred)
    valid_pred = np.maximum(valid_pred, 0)

    
    test_pred = model.predict(X_test)
    test_pred = np.expm1(test_pred)
    test_pred = np.maximum(test_pred, 0)

    
    y_true_valid = y.iloc[valid_index]
    score = np.sqrt(mean_squared_log_error(y_true_valid, valid_pred))
    cv_scores.append(score)

  
    oof_predictions[valid_index] = valid_pred
    test_predictions = test_predictions + (test_pred / 5)

    print("Fold RMSLE :", round(score, 5))

    fold_number = fold_number + 1"""

'fold_number = 1\n\nfor train_index, valid_index in kf.split(X):\n    print("\nFold", fold_number)\n\n    X_train = X.iloc[train_index]\n    X_valid = X.iloc[valid_index]\n\n    y_train = np.log1p(y.iloc[train_index])\n    y_valid = np.log1p(y.iloc[valid_index])\n\n    model = CatBoostRegressor(\n        iterations=5000,\n        learning_rate=0.03,\n        depth=8,\n        loss_function="RMSE",\n        random_seed=42,\n        early_stopping_rounds=200,\n        verbose=500\n    )\n\n    model.fit(\n        X_train,\n        y_train,\n        eval_set=(X_valid, y_valid),\n        use_best_model=True\n    )\n\n   \n    valid_pred = model.predict(X_valid)\n    valid_pred = np.expm1(valid_pred)\n    valid_pred = np.maximum(valid_pred, 0)\n\n    \n    test_pred = model.predict(X_test)\n    test_pred = np.expm1(test_pred)\n    test_pred = np.maximum(test_pred, 0)\n\n    \n    y_true_valid = y.iloc[valid_index]\n    score = np.sqrt(mean_squared_log_error(y_true_valid, valid_pred))\n    c

In [44]:
"""print("\nCross Validation Results")

for i in range(len(cv_scores)):
    print("Fold " + str(i+1) + ": " + str(round(cv_scores[i], 5)))

mean_score = sum(cv_scores) / len(cv_scores)
print("\nMean RMSLE :",round(mean_score, 5))"""

'print("\nCross Validation Results")\n\nfor i in range(len(cv_scores)):\n    print("Fold " + str(i+1) + ": " + str(round(cv_scores[i], 5)))\n\nmean_score = sum(cv_scores) / len(cv_scores)\nprint("\nMean RMSLE :",round(mean_score, 5))'

In [45]:
"""oof_preds_safe = np.maximum(oof_predictions, 0)
final_oof = np.sqrt(mean_squared_log_error(y, oof_preds_safe))
print("OOF RMSLE  :", round(final_oof, 5))"""

'oof_preds_safe = np.maximum(oof_predictions, 0)\nfinal_oof = np.sqrt(mean_squared_log_error(y, oof_preds_safe))\nprint("OOF RMSLE  :", round(final_oof, 5))'

In [46]:
"""submission = pd.DataFrame()
submission["TransactionID"] = test_ids
submission["TargetValue"] = test_predictions

submission.to_csv("submission.csv", index=False)
print(submission.head())"""

'submission = pd.DataFrame()\nsubmission["TransactionID"] = test_ids\nsubmission["TargetValue"] = test_predictions\n\nsubmission.to_csv("submission.csv", index=False)\nprint(submission.head())'